# TinyRAG Colab CUDA Demo

This notebook clones the TinyRAG repository, installs the CUDA llama.cpp runtime with `uv`, downloads a Qwen2.5 GGUF model, rebuilds the local RAG index, and runs grounded generation with GPU offload.

Before running all cells, set **Runtime > Change runtime type > GPU** in Colab.

In [ ]:
!nvidia-smi

## Clone the repository

In [ ]:
REPO_URL = "https://github.com/cxyfer/tinyrag-laptop-assistant.git"
BRANCH = "main"
PROJECT_DIR = "/content/tinyrag-laptop-assistant"

!rm -rf {PROJECT_DIR}
!git clone --branch {BRANCH} {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}

## Install the CUDA llama.cpp runtime

In [ ]:
!python -m pip install -q uv
!uv sync --frozen --extra llama-cu121

In [ ]:
!uv run --frozen --extra llama-cu121 python - <<'PY'
from llama_cpp import llama_cpp

print("gpu_offload_supported=", llama_cpp.llama_supports_gpu_offload())
PY

## Download the GGUF model

In [ ]:
!uvx --from huggingface-hub hf download \
  Qwen/Qwen2.5-1.5B-Instruct-GGUF \
  qwen2.5-1.5b-instruct-q4_k_m.gguf \
  --local-dir models

## Build the local RAG index

In [ ]:
!uv run --frozen --extra llama-cu121 tinyrag ingest --prefer-cache
!uv run --frozen --extra llama-cu121 tinyrag build-index

## Ask a grounded question

In [ ]:
!uv run --frozen --extra llama-cu121 tinyrag ask "BXH 使用哪一張顯示卡？" \
  --model-path models/qwen2.5-1.5b-instruct-q4_k_m.gguf \
  --n-ctx 2048 \
  --temperature 0.1 \
  --max-tokens 256 \
  --n-gpu-layers 35

## Optional: run the real-model benchmark

In [ ]:
!uv run --frozen --extra llama-cu121 tinyrag benchmark --use-model \
  --model-path models/qwen2.5-1.5b-instruct-q4_k_m.gguf \
  --n-ctx 2048 \
  --temperature 0.1 \
  --max-tokens 256 \
  --n-gpu-layers 35